In [9]:
from mlflow import MlflowClient
import mlflow
from pathlib import Path
import pandas as pd

In [10]:
data_yaml = "/workspace/NeuroVision_AI/notebooks/data.yaml"

In [11]:
runs_dir = "runs/detect"

run_names = [
    "brain_tumor_yolov8n-2",
    "brain_tumor_yolov8n-3",
]

model_name = "NeuroVision_BrainTumor"

# Maximum tolerated mAP50-95 drop
MAP_DROP_TOLERANCE = 0.02

# -------------------------
# Evaluate both models
# -------------------------
results = []

for run_name in run_names:

    model = YOLO(f"{runs_dir}/{run_name}/weights/best.pt")

    metrics = model.val(
        data= data_yaml,
        split="val",
        plots=True,
    )

    results.append({
        "run_name": run_name,
        "recall": metrics.box.mr,
        "precision": metrics.box.mp,
        "map50": metrics.box.map50,
        "map50_95": metrics.box.map,
    })

comparison = pd.DataFrame(results)


print(comparison)

Ultralytics 8.4.191 🚀 Python-3.10.12 torch-2.14.1+cu130 CUDA:0 (NVIDIA RTX A4000, 15974MiB)
Model summary (fused): 73 layers, 3,133,436 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 635.0±301.5 MB/s, size: 21.0 KB)
val: Scanning /workspace/dataset_yolo/labels/val.cache... 509 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 509/509 26.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 7.5it/s 4.3s<0.2s
                   all        509        553      0.628      0.593      0.657      0.424
                Glioma        135        153      0.592      0.359      0.412      0.256
            Meningioma        140        142      0.733      0.577      0.725      0.549
              No Tumor         97         97      0.497      0.794      0.782      0.431
             Pituitary        154        161      0.689       0.64      0.709      0.459
Speed: 1.4ms preprocess, 1.9ms 

In [12]:

# Among eligible models, choose the one with the highest recall
champion = comparison.sort_values(
    by="recall",
    ascending=False
).iloc[0]

champion_name = champion["run_name"]

print("Selected champion:", champion_name)
print("Recall:", champion["recall"])
print("mAP50:", champion["map50"])
print("mAP50-95:", champion["map50_95"])

Selected champion: brain_tumor_yolov8n-3
Recall: 0.8712976952454268
mAP50: 0.9418961312020341
mAP50-95: 0.7361962021435938


In [13]:
#getting the champion model path  to convert it to onnx 
best_pt_path = f"runs/detect/{champion_name}/weights/best.pt"

model = YOLO(best_pt_path)

onnx_path = model.export(format="onnx")

print("ONNX model:", onnx_path)



Ultralytics 8.4.191 🚀 Python-3.10.12 torch-2.14.1+cu130 CPU (AMD EPYC 7452 32-Core Processor)
Model summary (fused): 73 layers, 3,133,436 parameters, 0 gradients, 8.6 GFLOPs

PyTorch: starting from 'runs/detect/brain_tumor_yolov8n-3/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (6.2 MB)

ONNX: starting export with onnx 1.23.2 opset 18...
ONNX: slimming with onnxslim 0.1.98...
ONNX: export success ✅ 2.1s, saved as 'runs/detect/brain_tumor_yolov8n-3/weights/best.onnx' (12.2 MB)

Export complete (2.3s)
Results saved to /runs/detect/brain_tumor_yolov8n-3/weights/best.onnx
Predict:         yolo predict task=detect model=runs/detect/brain_tumor_yolov8n-3/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=runs/detect/brain_tumor_yolov8n-3/weights/best.onnx imgsz=640 data=/workspace/NeuroVision_AI/notebooks/data.yaml 
Visualize:       https://netron.app
ONNX model: runs/detect/brain_tumor_yolov8n-3/weights/best.onnx


In [15]:
import onnx
model_name = "NeuroVision_BrainTumor"

onnx_model = onnx.load(onnx_path)

with mlflow.start_run(run_name=champion_name):

    model_info = mlflow.onnx.log_model(
        onnx_model=onnx_model,
        name="onnx_model",
        registered_model_name=model_name
    )

    print( model_info.registered_model_version)

1


Successfully registered model 'NeuroVision_BrainTumor'.
Created version '1' of model 'NeuroVision_BrainTumor'.


In [23]:
client = MlflowClient()

version = model_info.registered_model_version

client.set_registered_model_alias(
    name=model_name,
    alias="champion",
    version=version
)

print(f"{model_name} version {version} is now champion!")

NeuroVision_BrainTumor version 1 is now champion!


In [24]:
champion_model = client.get_model_version_by_alias(
    model_name,
    "champion"
)

print("Champion version:", champion_model.version)
print("Model URI:", f"models:/{model_name}@champion")

Champion version: 1
Model URI: models:/NeuroVision_BrainTumor@champion
